# 03 — AI Career Mentor (RAG) Prototype

1. Load + chunk the career notes with `src.parsing.loader.load_folder('data/career_notes')`.
2. Embed the chunks and build a **FAISS** index; save it to `config.NOTES_INDEX_DIR`.
3. Build the RAG chain with **LangChain**: retrieve top-K chunks for a question, stuff them into the mentor prompt, generate an answer that stays inside the notes.
4. Test three questions: one answerable from the notes, one not (it must refuse / say "I don't know"), and one that needs two notes together.
5. Add the guardrail check (`src/safety/guardrails.py`) before the chain. Move the working chain into `src/mentor/rag_chain.py`.

In [ ]:
"""RAG-based AI Career Mentor for SmartHire."""

from pathlib import Path
import sys
import os

import faiss
import numpy as np
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate


# ============================================================
# PROJECT PATH
# ============================================================

PROJECT_ROOT = Path(__file__).resolve().parents[2]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


# ============================================================
# PROJECT IMPORTS
# ============================================================

from src import config
from src.generate.prompts import MENTOR_SYSTEM_PROMPT
from src.safety.guardrails import validate_question


# ============================================================
# ENVIRONMENT
# ============================================================

load_dotenv(PROJECT_ROOT / ".env")

api_key = os.getenv(config.API_KEY_ENV)

if not api_key:
    raise RuntimeError(
        f"{config.API_KEY_ENV} was not found in the environment."
    )


# ============================================================
# MODELS
# ============================================================

CHAT_MODEL = config.CHAT_MODEL

RAG_EMBED_MODEL = "all-MiniLM-L6-v2"
RAG_EMBED_DIM = 384

embedding_model = SentenceTransformer(RAG_EMBED_MODEL)


# ============================================================
# FAISS INDEX
# ============================================================

INDEX_PATH = config.NOTES_INDEX_DIR / "index.faiss"
METADATA_PATH = config.NOTES_INDEX_DIR / "notes_metadata.npy"


if not INDEX_PATH.exists():
    raise FileNotFoundError(
        f"Career notes FAISS index not found:\n{INDEX_PATH}"
    )

if not METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Career notes metadata not found:\n{METADATA_PATH}"
    )


index = faiss.read_index(str(INDEX_PATH))

metadata = np.load(
    str(METADATA_PATH),
    allow_pickle=True
)


# ============================================================
# VALIDATE INDEX
# ============================================================

if index.d != RAG_EMBED_DIM:
    raise ValueError(
        f"FAISS dimension mismatch. "
        f"Index dimension={index.d}, "
        f"expected={RAG_EMBED_DIM}."
    )


if index.ntotal != len(metadata):
    raise ValueError(
        f"FAISS/metadata count mismatch. "
        f"Index contains {index.ntotal} vectors, "
        f"but metadata contains {len(metadata)} records."
    )


# ============================================================
# LLM
# ============================================================

llm = ChatGoogleGenerativeAI(
    model=CHAT_MODEL,
    google_api_key=api_key,
    temperature=0.2,
)


prompt_template = ChatPromptTemplate.from_template(
    MENTOR_SYSTEM_PROMPT
)


# ============================================================
# HELPER: SAFE METADATA CONVERSION
# ============================================================

def _metadata_to_dict(item):
    """
    Convert a metadata item into a normal dictionary.

    This prevents errors such as:
        'NoneType' object has no attribute 'get'
    """

    if item is None:
        return {}

    if isinstance(item, dict):
        return item

    if hasattr(item, "item"):
        try:
            value = item.item()

            if isinstance(value, dict):
                return value

        except Exception:
            pass

    return {}


# ============================================================
# RETRIEVE CAREER NOTES
# ============================================================

def retrieve_notes(question: str, top_k: int = config.TOP_K_NOTES):
    """
    Retrieve the most relevant career-note chunks.
    """

    if not isinstance(question, str):
        raise TypeError("Question must be a string.")

    question = question.strip()

    if not question:
        return []

    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )

    if query_embedding.shape[1] != index.d:
        raise ValueError(
            f"Query embedding dimension {query_embedding.shape[1]} "
            f"does not match FAISS dimension {index.d}."
        )

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx < 0 or idx >= len(metadata):
            continue

        raw_item = metadata[idx]

        item = _metadata_to_dict(raw_item)

        if not item:
            continue

        text = (
            item.get("text")
            or item.get("content")
            or item.get("chunk")
            or ""
        )

        source = (
            item.get("source")
            or item.get("filename")
            or item.get("file")
            or "Unknown source"
        )

        text = str(text).strip()
        source = str(source).strip()

        if not text:
            continue

        results.append(
            {
                "score": float(score),
                "text": text,
                "source": source,
            }
        )

    return results


# ============================================================
# FORMAT CONTEXT
# ============================================================

def format_context(results):
    """
    Convert retrieved notes into the context sent to Gemini.
    """

    if not results:
        return "No relevant career notes were retrieved."

    context_parts = []

    for i, result in enumerate(results, start=1):

        if result is None:
            continue

        text = result.get("text", "")
        source = result.get("source", "Unknown source")

        if not text:
            continue

        context_parts.append(
            f"[Source {i}: {source}]\n{text}"
        )

    if not context_parts:
        return "No relevant career notes were retrieved."

    return "\n\n".join(context_parts)


# ============================================================
# CAREER MENTOR
# ============================================================

def career_mentor(question: str):
    """
    Answer a career question using retrieved career notes.

    Returns:
        {
            "answer": str,
            "sources": list
        }
    """

    # --------------------------------------------------------
    # 1. GUARDRAIL CHECK
    # --------------------------------------------------------

    allowed, reason = validate_question(question)

    if not allowed:
        return {
            "answer": f"Question rejected by safety guardrails: {reason}",
            "sources": [],
        }


    # --------------------------------------------------------
    # 2. RETRIEVE CAREER NOTES
    # --------------------------------------------------------

    retrieved = retrieve_notes(
        question,
        top_k=config.TOP_K_NOTES
    )


    # --------------------------------------------------------
    # 3. NO RETRIEVED INFORMATION
    # --------------------------------------------------------

    if not retrieved:

        return {
            "answer": "I don't know based on the provided career notes.",
            "sources": [],
        }


    # --------------------------------------------------------
    # 4. BUILD CONTEXT
    # --------------------------------------------------------

    context = format_context(retrieved)


    # --------------------------------------------------------
    # 5. CREATE PROMPT
    # --------------------------------------------------------

    messages = prompt_template.format_messages(
        context=context,
        question=question,
    )


    # --------------------------------------------------------
    # 6. CALL GEMINI
    # --------------------------------------------------------

    response = llm.invoke(messages)


    # --------------------------------------------------------
    # 7. EXTRACT ANSWER SAFELY
    # --------------------------------------------------------

    answer = getattr(response, "content", None)

    if answer is None:
        answer = str(response)

    if isinstance(answer, list):

        parts = []

        for item in answer:

            if isinstance(item, dict):
                text = item.get("text")

                if text:
                    parts.append(str(text))

            elif isinstance(item, str):
                parts.append(item)

        answer = "\n".join(parts)


    answer = str(answer).strip()


    if not answer:
        answer = (
            "I don't know based on the provided career notes."
        )


    # --------------------------------------------------------
    # 8. RETURN SOURCES
    # --------------------------------------------------------

    sources = []

    for item in retrieved:

        if item is None:
            continue

        source = item.get("source")

        if source and source not in sources:
            sources.append(source)


    return {
        "answer": answer,
        "sources": sources,
    }


# ============================================================
# SIMPLE ASK FUNCTION
# ============================================================

def ask_mentor(question: str):
    """
    Simple wrapper for the career mentor.
    """

    return career_mentor(question)


# ============================================================
# TEST
# ============================================================

if __name__ == "__main__":

    print("=" * 60)
    print("SMART HIRE - AI CAREER MENTOR")
    print("=" * 60)

    test_questions = [
        "What skills should I learn for a career in data science?",
        "How can I improve my career based on the available career notes?",
        "What is the salary of a software engineer at Google?",
    ]

    for question in test_questions:

        print()
        print("-" * 60)
        print("QUESTION:")
        print(question)
        print("-" * 60)

        try:

            result = career_mentor(question)

            print()
            print("ANSWER:")
            print(result["answer"])

            print()
            print("SOURCES:")

            if result["sources"]:
                for source in result["sources"]:
                    print(f"- {source}")
            else:
                print("None")

        except Exception as e:

            print()
            print("ERROR:")
            print(type(e).__name__, "-", e)

    print()
    print("=" * 60)
    print("CAREER MENTOR TEST COMPLETED")
    print("=" * 60)

Project folder: c:\Users\koppa\SmartHire-GenAI
Gemini API key loaded successfully.
Career Mentor model: gemini-3.5-flash-lite

Career notes folder:
C:\Users\koppa\SmartHire-GenAI\data\career_notes

CAREER NOTES LOADED
Total chunks: 5

First career-note chunk:
----------------------------------------------------------------------
Source: data_analyst_roadmap.md
Text:
# How to Become a Data Analyst A data analyst collects, cleans, and interprets data to help a business make decisions. It is one of the most common first roles for people moving into data. ## Core skills - **SQL** — the single most important skill. You must be able to write SELECT queries with JOINs, GROUP BY, and filtering. - **Spreadsheets** — Excel or Google Sheets: pivot tables, lookups, charts. - **A visualisation tool** — Power BI or Tableau to build dashboards. - **Statistics basics** — averages, distributions, correlation, and how to spot a misleading chart. - **Python (optional but valued)** — pandas for cleaning a

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2764.09it/s]


Local embedding model loaded successfully.

Total note chunks: 5

CREATING CAREER NOTE EMBEDDINGS


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.65it/s]



Career-note embeddings created successfully.
Number of chunks: 5
Embedding shape: (5, 384)
Embedding dimension: 384

CREATING FAISS INDEX
FAISS index created successfully.
Vectors stored: 5

FAISS index saved to:
C:\Users\koppa\SmartHire-GenAI\vectorstore\notes_faiss\index.faiss

Career-note metadata saved to:
C:\Users\koppa\SmartHire-GenAI\vectorstore\notes_faiss\notes_metadata.npy

CREATING GEMINI CAREER MENTOR
Gemini Career Mentor created successfully.
Model: gemini-3.5-flash-lite
Career Mentor prompt created successfully.
Guardrail check ready.
Career-note retriever ready.
Career Mentor RAG chain ready.

TEST 1 — ANSWERABLE QUESTION

Question:
What skills should I learn to become an AI Engineer?


...


Answer:
[{'type': 'text', 'text': "I don't know based on the provided career notes.", 'extras': {'signature': 'El4KXAERTTIPrJx5N2KUblv6fo7Ch1OleGRT22QdEsh4y95H+y1y8xpF0JhODqUsSudIbdVEa1i3K8MpgVEBeldsHj6ZA6NdLFJPh9sJCRfI8xDyiaJneu96iuSGpGxH'}}]

Retrieved Sources:
- data_analyst_roadmap.md | similarity=0.3421
- data_analyst_roadmap.md | similarity=0.3123
- resume_writing_tips.md | similarity=0.2558

TEST 2 — NOT ANSWERABLE QUESTION

Question:
What is the current stock price of Apple?


c:\Users\koppa\SmartHire-GenAI\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Answer:
[{'type': 'text', 'text': "I don't know based on the provided career notes.", 'extras': {'signature': 'El4KXAERTTIPfZBc/QCUseStIO+fJcLH5EOubpY+h6ZmXIGe16rXdKAvvVp9KPOHYxf4kYZhOT3LJ3bTUfaQBTC81x/7UBhyzbGZsuQS2JZBtrRGMgcRYyPw/SIc5oLh'}}]

Retrieved Sources:
- data_analyst_roadmap.md | similarity=0.1758
- data_analyst_roadmap.md | similarity=0.0833
- data_analyst_roadmap.md | similarity=0.0680

TEST 3 — MULTIPLE NOTES

Question:
How can I prepare myself for an AI Engineer career, including the skills I should learn and the career development steps I should follow?


c:\Users\koppa\SmartHire-GenAI\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Answer:
[{'type': 'text', 'text': "I don't know based on the provided career notes.", 'extras': {'signature': 'El4KXAERTTIP5bi3rGV7Ro/jdBuuhJrGf2I6DRa92/xkG9Ff6Qcnnwo6sEJ8jsZvMbOdWqhekzwq3MrJiGlx6ewITq2ycU25wpzzFTyzVUQo29TxkPArDQgJolFOrKIw'}}]

Retrieved Sources:
- data_analyst_roadmap.md | similarity=0.3866
- data_analyst_roadmap.md | similarity=0.3784
- resume_writing_tips.md | similarity=0.3364

SMART HIRE — CAREER MENTOR RAG COMPLETED

Career-note chunks: 5
Embedding dimension: 384
FAISS vectors: 5
Top-K notes: 3

FAISS index:
C:\Users\koppa\SmartHire-GenAI\vectorstore\notes_faiss\index.faiss

Metadata:
C:\Users\koppa\SmartHire-GenAI\vectorstore\notes_faiss\notes_metadata.npy

RAG PIPELINE:
User Question
      ↓
Guardrail
      ↓
Local Embedding
      ↓
FAISS Retrieval
      ↓
Top-K Career Notes
      ↓
LangChain
      ↓
Gemini 3.5 Flash Lite
      ↓
Grounded Career Answer

SUCCESS


In [ ]:
# ============================================================
# SMART HIRE - NOTEBOOK 03
# BUILD CAREER NOTES FAISS INDEX
# ============================================================

from pathlib import Path
import sys
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

from pathlib import Path
import sys

# Find the SmartHire project root
PROJECT_ROOT = Path.cwd().parent

# Add project root to Python path
sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.parsing.loader import load_text, chunk_text

print("Project root:", PROJECT_ROOT)
print("src found:", (PROJECT_ROOT / "src").exists())


# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path.cwd()

# Make sure project root is available for imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CAREER_NOTES_DIR = config.CAREER_NOTES_DIR
NOTES_INDEX_DIR = config.NOTES_INDEX_DIR

print("=" * 70)
print("SMART HIRE - CAREER NOTES INDEXING")
print("=" * 70)

print("\nCareer notes folder:")
print(CAREER_NOTES_DIR)

print("\nFAISS output folder:")
print(NOTES_INDEX_DIR)


# ============================================================
# 2. FIND ALL CAREER NOTE FILES
#    rglob() includes Career_Guides and its subfolders
# ============================================================

SUPPORTED_EXTENSIONS = {
    ".txt",
    ".md",
    ".pdf",
    ".docx",
}

note_files = sorted(
    path
    for path in CAREER_NOTES_DIR.rglob("*")
    if path.is_file()
    and path.suffix.lower() in SUPPORTED_EXTENSIONS
)

print("\n" + "=" * 70)
print("FILES FOUND")
print("=" * 70)

print(f"Total files found: {len(note_files)}")

for path in note_files:
    print(
        " -",
        path.relative_to(CAREER_NOTES_DIR)
    )


# ============================================================
# 3. LOAD ALL DOCUMENTS
# ============================================================

documents = []

print("\n" + "=" * 70)
print("LOADING DOCUMENTS")
print("=" * 70)

for path in note_files:

    try:

        text = load_text(path)

        if text and text.strip():

            documents.append(
                {
                    "text": text,
                    "source": path.name,
                    "path": str(path),
                }
            )

            print(
                f"✓ Loaded: "
                f"{path.relative_to(CAREER_NOTES_DIR)}"
            )

        else:

            print(
                f"⚠ Empty file: "
                f"{path.name}"
            )

    except Exception as e:

        print(
            f"❌ Could not read "
            f"{path.name}: {e}"
        )


print(f"\nDocuments successfully loaded: {len(documents)}")


# ============================================================
# 4. CREATE CHUNKS
# ============================================================

chunks = []

print("\n" + "=" * 70)
print("CREATING CHUNKS")
print("=" * 70)

for document in documents:

    document_chunks = chunk_text(
        document["text"],
        chunk_size=config.CHUNK_SIZE,
        overlap=config.CHUNK_OVERLAP,
    )

    for chunk in document_chunks:

        chunks.append(
            {
                "text": chunk,
                "source": document["source"],
                "path": document["path"],
            }
        )


print(f"Total chunks created: {len(chunks)}")


# ============================================================
# 5. CHECK THAT CHUNKS EXIST
# ============================================================

if not chunks:

    raise ValueError(
        "No career-note chunks were created. "
        "Check data/career_notes/ and make sure it "
        "contains .txt, .md, .pdf, or .docx files."
    )


# ============================================================
# 6. LOAD EMBEDDING MODEL
#    MUST MATCH THE EXISTING RAG INDEX
# ============================================================

EMBED_MODEL = "all-MiniLM-L6-v2"
EMBED_DIM = 384

print("\n" + "=" * 70)
print("LOADING EMBEDDING MODEL")
print("=" * 70)

print("Model:", EMBED_MODEL)
print("Dimension:", EMBED_DIM)

embedding_model = SentenceTransformer(
    EMBED_MODEL
)

print("✓ Embedding model loaded")


# ============================================================
# 7. CREATE EMBEDDINGS
# ============================================================

texts = [
    item["text"]
    for item in chunks
]

print("\n" + "=" * 70)
print("CREATING EMBEDDINGS")
print("=" * 70)

embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)

embeddings = np.asarray(
    embeddings,
    dtype=np.float32,
)

print("\nEmbedding shape:")
print(embeddings.shape)


# ============================================================
# 8. CHECK EMBEDDING DIMENSION
# ============================================================

if embeddings.shape[1] != EMBED_DIM:

    raise ValueError(
        f"Embedding dimension mismatch. "
        f"Got {embeddings.shape[1]}, "
        f"expected {EMBED_DIM}."
    )

print("✓ Embedding dimension is correct")


# ============================================================
# 9. CREATE FAISS INDEX
# ============================================================

print("\n" + "=" * 70)
print("CREATING FAISS INDEX")
print("=" * 70)

index = faiss.IndexFlatIP(
    EMBED_DIM
)

index.add(
    embeddings
)

print("FAISS vectors:", index.ntotal)
print("FAISS dimension:", index.d)


# ============================================================
# 10. CREATE METADATA
#     RAG CHAIN USES text + source + path
# ============================================================

metadata = np.array(
    [
        {
            "text": item["text"],
            "source": item["source"],
            "path": item["path"],
        }
        for item in chunks
    ],
    dtype=object,
)

print("\nMetadata records:", len(metadata))


# ============================================================
# 11. VALIDATE INDEX
# ============================================================

print("\n" + "=" * 70)
print("VALIDATING INDEX")
print("=" * 70)

if index.ntotal != len(metadata):

    raise ValueError(
        "FAISS index count and metadata count do not match."
    )

if index.d != EMBED_DIM:

    raise ValueError(
        "FAISS index dimension does not match "
        "embedding dimension."
    )

print("✓ Vector count matches metadata count")
print("✓ Embedding dimension matches")


# ============================================================
# 12. CREATE OUTPUT DIRECTORY
# ============================================================

NOTES_INDEX_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 13. SAVE FAISS INDEX
# ============================================================

index_path = (
    NOTES_INDEX_DIR / "index.faiss"
)

metadata_path = (
    NOTES_INDEX_DIR / "notes_metadata.npy"
)

print("\n" + "=" * 70)
print("SAVING INDEX")
print("=" * 70)

faiss.write_index(
    index,
    str(index_path),
)

np.save(
    str(metadata_path),
    metadata,
    allow_pickle=True,
)

print("✓ FAISS index saved:")
print(index_path)

print("\n✓ Metadata saved:")
print(metadata_path)


# ============================================================
# 14. LOAD SAVED FILES AGAIN FOR FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL VERIFICATION")
print("=" * 70)

saved_index = faiss.read_index(
    str(index_path)
)

saved_metadata = np.load(
    str(metadata_path),
    allow_pickle=True,
)

print("Saved FAISS vectors:")
print(saved_index.ntotal)

print("\nSaved metadata records:")
print(len(saved_metadata))

print("\nSaved FAISS dimension:")
print(saved_index.d)

print("\nExpected dimension:")
print(EMBED_DIM)


# ============================================================
# 15. FINAL CHECK
# ============================================================

if (
    saved_index.ntotal == len(saved_metadata)
    and saved_index.d == EMBED_DIM
):

    print("\n" + "=" * 70)
    print("✅ NOTEBOOK 03 COMPLETED SUCCESSFULLY")
    print("=" * 70)

    print(
        f"✅ Files indexed: {len(note_files)}"
    )

    print(
        f"✅ Chunks indexed: {len(chunks)}"
    )

    print(
        f"✅ FAISS vectors: {saved_index.ntotal}"
    )

    print(
        "✅ Career_Guides folder included"
    )

    print(
        "✅ RAG-compatible index created"
    )

    print("\nIndex location:")
    print(index_path)

    print("\nMetadata location:")
    print(metadata_path)

else:

    print("\n" + "=" * 70)
    print("❌ INDEX VERIFICATION FAILED")
    print("=" * 70)

    raise ValueError(
        "The saved FAISS index could not be verified."
    )

c:\Users\koppa\SmartHire-GenAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
